# Frequency tagging of LLM hidden states

Computes the target-frequency z-scores and the structured-versus-reversed tests
reported in the manuscript. Run top to bottom on a GPU runtime; extraction is
cached to Drive, so later runs need no GPU. `figures.py` draws the figures from
the same cache.

    corpus text -> units (Chinese character / English word)
      -> tokenize -> forward pass -> hidden states, embedding layer dropped
      -> average the tokens covering each unit          [layers, dim, positions]
      -> Frobenius norm over layers and dims            -> magnitude spectrum
      -> z-score at 0.25 and 0.5 unit^-1

## 1. Environment

GPU and versions of the published run: `ENVIRONMENT.md`.

In [ ]:
# torch, numpy and scipy come with the Colab image and are left unpinned.
!pip -q install "transformers==5.13.1" "rwkv-fla==0.7.202508221413"

In [ ]:
# Environment record for ENVIRONMENT.md.
import subprocess, sys
print(subprocess.run([sys.executable, "-m", "pip", "list"], capture_output=True,
                     text=True).stdout)
!nvidia-smi --query-gpu=name,driver_version --format=csv,noheader

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json
import re
from pathlib import Path

import numpy as np

# Not redistributed; see data/README.md.
CORPUS_DIR = Path("/content/drive/MyDrive/llm-grammatical-oscillations/corpora")
# Cached activations and result tables; figures.py reads the activations.
OUTPUT_DIR = Path("/content/drive/MyDrive/llm-grammatical-oscillations/output")

## 2. Settings

In [ ]:
SENTENCE_FREQ = 0.25  # unit^-1
PHRASE_FREQ = 0.50  # unit^-1
Z_THRESHOLD = 3.0
SENTENCE_LEN = 4  # units per sentence

# Segment length for the structured-vs-reversed test; a multiple of the
# sentence period, so both targets fall on DFT bins of every segment.
SEGMENT_LEN = 20

# `tokenizer`: from_pretrained arguments that put a special token at position 0.
MODELS = {
    "gpt2-zh": {
        "hf": "uer/gpt2-large-chinese-cluecorpussmall",
        "label": "GPT-2",
        # BertTokenizer adds [CLS] by default.
        "tokenizer": {},
    },
    "rwkv7": {
        "hf": "fla-hub/rwkv7-1.5B-world",
        "label": "RWKV-7",
        # Without this, add_special_tokens=True adds nothing.
        "tokenizer": {"add_bos_token": True},
    },
}

# `source`: a corpus file, or a derivation from Chinese-sentences.txt.
CORPORA = {
    "random": {
        "language": "zh",
        "source": "Chinese-random.txt",
        "label": "Random",
    },
    "phrase": {
        "language": "zh",
        "source": "derive:phrase_only",
        "label": "Phrase only",
    },
    "phrase+sentence": {
        "language": "zh",
        "source": "Chinese-sentences.txt",
        "label": "Phrase + sentence",
    },
    "reversed": {
        "language": "zh",
        "source": "derive:reversed_phrases",
        "label": "Reversed phrases",
    },
    "english": {
        "language": "en",
        "source": "English-sentences.txt",
        "label": "English",
    },
}

## 3. Corpora

In [ ]:
def read_units(path, language):
    """Chinese: characters, whitespace removed. English: whitespace-separated words."""
    text = Path(path).read_text(encoding="utf-8")
    if language == "en":
        return text.split()
    return list(re.sub(r"\s+", "", text))


def phrase_only(units):
    """First two units of every four-unit sentence."""
    return [u for i in range(0, len(units), SENTENCE_LEN)
            for u in units[i:i + SENTENCE_LEN // 2]]


def reversed_phrases(units):
    """Swap the two units inside each phrase (Lo et al. 2022)."""
    out = []
    for i in range(0, len(units), SENTENCE_LEN):
        first, second, third, fourth = units[i:i + SENTENCE_LEN]
        out += [second, first, fourth, third]
    return out


DERIVATIONS = {"phrase_only": phrase_only, "reversed_phrases": reversed_phrases}


def build_corpora(corpus_dir):
    """All five corpora as lists of units, keyed as in CORPORA."""
    corpus_dir = Path(corpus_dir)
    sentences = read_units(corpus_dir / "Chinese-sentences.txt", "zh")
    if len(sentences) % SENTENCE_LEN:
        raise ValueError(
            f"Chinese-sentences.txt holds {len(sentences)} characters, "
            f"not a whole number of {SENTENCE_LEN}-character sentences"
        )

    corpora = {}
    for name, spec in CORPORA.items():
        source = spec["source"]
        if source.startswith("derive:"):
            corpora[name] = DERIVATIONS[source.removeprefix("derive:")](sentences)
        else:
            corpora[name] = read_units(corpus_dir / source, spec["language"])
    return corpora

## 4. Signal and spectrum

In [ ]:
def global_signal(activations):
    """Frobenius norm over layers and dims, one value per position."""
    return np.sqrt(np.sum(activations.astype(np.float64) ** 2, axis=(0, 1)))


def layer_signals(activations):
    """Per-layer L2 norm, shape [layers, positions]."""
    return np.linalg.norm(activations.astype(np.float64), axis=1)


def spectrum(signal):
    """Magnitude spectrum of a signal sampled once per position."""
    return np.fft.rfftfreq(len(signal), d=1.0), np.abs(np.fft.rfft(signal))


def z_at(freqs, mags, target, exclude=()):
    """Target bin against the mean and s.d. of the other bins, DC excluded."""
    index = int(np.argmin(np.abs(freqs - target)))
    drop = {0, index, *(int(np.argmin(np.abs(freqs - f))) for f in exclude)}
    noise = np.array([mags[i] for i in range(1, len(mags)) if i not in drop])
    if noise.size == 0 or noise.std() == 0:
        return 0.0
    return float((mags[index] - noise.mean()) / noise.std())


def target_z_scores(signal):
    """(z at 0.25, z at 0.5); each baseline excludes the other target."""
    freqs, mags = spectrum(signal)
    return (
        z_at(freqs, mags, SENTENCE_FREQ, exclude=(PHRASE_FREQ,)),
        z_at(freqs, mags, PHRASE_FREQ, exclude=(SENTENCE_FREQ,)),
    )


def layerwise_z_scores(activations):
    """Per-layer (z at 0.25, z at 0.5), each shape [layers]."""
    scores = np.array([target_z_scores(s) for s in layer_signals(activations)])
    return scores[:, 0], scores[:, 1]

## 5. Extraction

In [ ]:
def load_model(hf_name, device, tokenizer_kwargs=None):
    """Model in float32 and eval mode, and its tokenizer."""
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(
        hf_name, trust_remote_code=True, **(tokenizer_kwargs or {})
    )
    model = AutoModelForCausalLM.from_pretrained(hf_name, trust_remote_code=True)
    return model.to(device=device, dtype=torch.float32).eval(), tokenizer


def manual_offsets(tokenizer, input_ids, text):
    """Character span of every token; special tokens get (0, 0) and match no unit."""
    special = set(tokenizer.all_special_ids)
    spans, pos = [], 0
    for token_id in input_ids:
        if token_id in special:
            spans.append((0, 0))
            continue
        piece = tokenizer.decode([token_id])
        start = text.find(piece, pos)
        if start < 0:  # decoding is not exactly invertible for this token
            start = pos
        spans.append((start, start + len(piece)))
        pos = start + len(piece)
    return spans


def unit_token_indices(units, text, spans, language):
    """For each unit, the indices of the tokens overlapping its span."""
    indices, pos = [], 0
    for unit in units:
        found = text.find(unit, pos)
        start = found if found >= 0 else pos
        end = start + len(unit)
        indices.append([i for i, (s, e) in enumerate(spans) if start < e and s < end])
        pos = end
        if language == "en":
            while pos < len(text) and text[pos].isspace():
                pos += 1
    return indices


def extract_activations(model, tokenizer, units, language, device):
    """Unit-aligned hidden states, shape [layers, dim, positions]."""
    import torch

    text = " ".join(units) if language == "en" else "".join(units)
    encoded = tokenizer(
        text, add_special_tokens=True, return_offsets_mapping=tokenizer.is_fast
    )
    input_ids = encoded["input_ids"]
    leading_special = input_ids[0] in set(tokenizer.all_special_ids)
    if not leading_special:
        print("    WARNING: no leading special token; position 0 is a unit")
    spans = (
        encoded["offset_mapping"]
        if tokenizer.is_fast
        else manual_offsets(tokenizer, input_ids, text)
    )

    with torch.no_grad():
        outputs = model(
            torch.tensor([input_ids], device=device),
            output_hidden_states=True,
            use_cache=False,
        )
    # hidden_states[0] is the embedding output.
    hidden = torch.stack(outputs.hidden_states[1:], dim=0)[:, 0]

    n_layers, _, n_dim = hidden.shape
    per_unit, unmatched = [], 0
    for indices in unit_token_indices(units, text, spans, language):
        if indices:
            per_unit.append(hidden[:, indices, :].mean(dim=1).float().cpu().numpy())
        else:
            unmatched += 1
            per_unit.append(np.zeros((n_layers, n_dim), dtype=np.float32))
    if unmatched:
        print(f"    WARNING: {unmatched}/{len(units)} units matched no token")

    # [layers, positions, dim] -> [layers, dim, positions]
    return (
        np.stack(per_unit, axis=1).transpose(0, 2, 1),
        unmatched,
        leading_special,
    )


def sink_ratio(activations):
    """Norm at position 0 over the median norm across positions."""
    x = global_signal(activations)
    return float(x[0] / np.median(x))


def cached_activations(model_key, corpus_key, corpora, output_dir, device):
    """Extract, or reload a previous extraction from disk."""
    path = Path(output_dir) / "activations" / model_key / f"{corpus_key}.npy"
    if path.exists():
        print(f"  {model_key} x {corpus_key}: cached")
        return np.load(path)

    model, tokenizer = load_model(
        MODELS[model_key]["hf"], device, MODELS[model_key].get("tokenizer")
    )
    activations, unmatched, leading_special = extract_activations(
        model, tokenizer, corpora[corpus_key], CORPORA[corpus_key]["language"], device
    )
    ratio = sink_ratio(activations)
    print(
        f"  {model_key} x {corpus_key}: {activations.shape} "
        f"position-0/median = {ratio:.2f}"
    )
    if ratio > 2.0:
        print("    WARNING: attention sink is on the first unit, not a special token")

    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, activations)
    (path.parent / f"{corpus_key}.json").write_text(
        json.dumps(
            {
                "model": MODELS[model_key]["hf"],
                "corpus": corpus_key,
                "language": CORPORA[corpus_key]["language"],
                "units": len(corpora[corpus_key]),
                "shape": list(activations.shape),
                "sink_ratio": ratio,
                "unmatched_units": unmatched,
                "leading_special_token": leading_special,
            },
            indent=2,
        )
    )
    return activations

## 6. Run

In [ ]:
import torch  # noqa: E402

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

corpora = build_corpora(CORPUS_DIR)
for name, units in corpora.items():
    print(f"{name:18s} {len(units):4d} units, {len(units) // SENTENCE_LEN} sentences")

activations = {}
for model_key in MODELS:
    for corpus_key in CORPORA:
        activations[model_key, corpus_key] = cached_activations(
            model_key, corpus_key, corpora, OUTPUT_DIR, DEVICE
        )

## 7. Target-frequency z-scores

In [ ]:
signals = {key: global_signal(a) for key, a in activations.items()}
z_scores = {key: target_z_scores(x) for key, x in signals.items()}

rows = []
for model_key, model in MODELS.items():
    for corpus_key, corpus in CORPORA.items():
        z_sentence, z_phrase = z_scores[model_key, corpus_key]
        rows.append(
            {
                "model": model["label"],
                "model_key": model_key,
                "corpus": corpus["label"],
                "corpus_key": corpus_key,
                "z_sentential": round(z_sentence, 3),
                "z_phrasal": round(z_phrase, 3),
            }
        )

print(f"{'model':8s} {'corpus':20s} {'z(0.25)':>9s} {'z(0.50)':>9s}")
for row in rows:
    mark = lambda z: "*" if z > Z_THRESHOLD else " "  # noqa: E731
    print(
        f"{row['model']:8s} {row['corpus']:20s} "
        f"{row['z_sentential']:8.2f}{mark(row['z_sentential'])} "
        f"{row['z_phrasal']:8.2f}{mark(row['z_phrasal'])}"
    )

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
(Path(OUTPUT_DIR) / "target_z_scores.json").write_text(json.dumps(rows, indent=2))

### Bins above threshold

In [ ]:
def suprathreshold_bins(signal, threshold=Z_THRESHOLD):
    """Every bin with z above threshold, against a baseline excluding both targets."""
    freqs, mags = spectrum(signal)
    scored = [
        (float(freqs[i]),
         z_at(freqs, mags, freqs[i], exclude=(SENTENCE_FREQ, PHRASE_FREQ)))
        for i in range(1, len(freqs))
    ]
    return [(freq, z) for freq, z in scored if z > threshold]


print(f"{'model':10s} {'corpus':18s}  bins with z > {Z_THRESHOLD}")
for model_key in MODELS:
    for corpus_key in CORPORA:
        hits = suprathreshold_bins(signals[model_key, corpus_key])
        listed = ", ".join(f"{freq:.3f} (z={z:.1f})" for freq, z in hits) or "none"
        print(f"{model_key:10s} {corpus_key:18s}  {listed}")

## 8. Structured versus reversed phrases

In [ ]:
def segment_snr(signal, target, segment_len=SEGMENT_LEN):
    """Response at `target` per segment, as a ratio to the non-target bins."""
    snrs = []
    for k in range(len(signal) // segment_len):
        freqs, mags = spectrum(signal[k * segment_len:(k + 1) * segment_len])
        index = int(np.argmin(np.abs(freqs - target)))
        drop = {
            0,
            int(np.argmin(np.abs(freqs - SENTENCE_FREQ))),
            int(np.argmin(np.abs(freqs - PHRASE_FREQ))),
        }
        noise = [mags[i] for i in range(1, len(mags)) if i not in drop]
        snrs.append(mags[index] / np.mean(noise))
    return np.array(snrs)


# (corpus, frequency, label), each compared with the reversed-phrases corpus.
COMPARISONS = [
    ("phrase", PHRASE_FREQ, "0.5"),
    ("phrase+sentence", PHRASE_FREQ, "0.5"),
    ("phrase+sentence", SENTENCE_FREQ, "0.25"),
]

In [ ]:
from scipy import stats  # noqa: E402


def compare_with_control(model_key, segment_len=SEGMENT_LEN):
    """(U, p) per comparison for one model, p uncorrected."""
    control = {
        freq: segment_snr(signals[model_key, "reversed"], freq, segment_len)
        for freq in (SENTENCE_FREQ, PHRASE_FREQ)
    }
    out = []
    for corpus_key, freq, freq_label in COMPARISONS:
        snr = segment_snr(signals[model_key, corpus_key], freq, segment_len)
        # "exact" for every n, rather than scipy's size-dependent default.
        u_statistic, p_value = stats.mannwhitneyu(
            snr, control[freq], alternative="greater", method="exact"
        )
        out.append((corpus_key, freq, freq_label, snr, control[freq],
                    float(u_statistic), float(p_value)))
    return out


results = [(model_key, *row) for model_key in MODELS
           for row in compare_with_control(model_key)]
# Benjamini-Hochberg across the six comparisons.
corrected = stats.false_discovery_control([row[-1] for row in results], method="bh")

stat_rows = []
for (model_key, corpus_key, freq, freq_label, snr, control, u, p), q in zip(
    results, corrected
):
    stat_rows.append(
        {
            "model": MODELS[model_key]["label"],
            "corpus": CORPORA[corpus_key]["label"],
            "frequency": freq_label,
            "n": len(snr),
            "n_control": len(control),
            "snr_mean": round(float(snr.mean()), 3),
            "snr_sd": round(float(snr.std(ddof=1)), 3),
            "control_mean": round(float(control.mean()), 3),
            "control_sd": round(float(control.std(ddof=1)), 3),
            "U": u,
            "p_uncorrected": p,
            "p": float(q),
        }
    )

for row in stat_rows:
    print(
        f"{row['model']:8s} {row['corpus']:20s} @{row['frequency']:>5s} unit^-1  "
        f"U={row['U']:5.1f} p={row['p']:.1e} (raw {row['p_uncorrected']:.1e})  "
        f"SNR {row['snr_mean']:5.2f}+-{row['snr_sd']:4.2f} (n={row['n']}) vs "
        f"{row['control_mean']:5.2f}+-{row['control_sd']:4.2f} (n={row['n_control']})"
    )

(Path(OUTPUT_DIR) / "structured_vs_reversed.json").write_text(
    json.dumps(stat_rows, indent=2)
)

### Segment length

In [ ]:
SEGMENT_LENGTHS = [12, 16, 20, 24]

by_length = {}
for length in SEGMENT_LENGTHS:
    runs = [(model_key, row) for model_key in MODELS
            for row in compare_with_control(model_key, length)]
    corrected = stats.false_discovery_control([row[-1] for _, row in runs],
                                              method="bh")
    for (model_key, row), q in zip(runs, corrected):
        by_length[length, model_key, row[0], row[2]] = float(q)

header = "  ".join(f"L={length:<2d}" for length in SEGMENT_LENGTHS)
print(f"{'model':8s} {'corpus':18s} {'freq':>5s}   {header}   (Benjamini-Hochberg P)")
for model_key in MODELS:
    for corpus_key, _, freq_label in COMPARISONS:
        cells = "  ".join(
            f"{by_length[length, model_key, corpus_key, freq_label]:.1e}"
            + ("*" if by_length[length, model_key, corpus_key, freq_label] < 0.05
               else " ")
            for length in SEGMENT_LENGTHS
        )
        print(f"{MODELS[model_key]['label']:8s} {CORPORA[corpus_key]['label']:18s} "
              f"{freq_label:>5s}   {cells}")

print("\n* P < 0.05")

## 9. Reproduction check

`REFERENCE_Z` holds the z-scores of the published run.

In [ ]:
REFERENCE_Z = {
    ("gpt2-zh", "random"): (-0.260, -1.403),
    ("gpt2-zh", "phrase"): (-0.200, 11.051),
    ("gpt2-zh", "phrase+sentence"): (9.043, 11.823),
    ("gpt2-zh", "reversed"): (1.782, 0.894),
    ("gpt2-zh", "english"): (1.292, 2.013),
    ("rwkv7", "random"): (0.260, -1.578),
    ("rwkv7", "phrase"): (-0.288, 43.991),
    ("rwkv7", "phrase+sentence"): (14.189, 56.664),
    ("rwkv7", "reversed"): (3.187, 0.252),
    ("rwkv7", "english"): (8.647, 11.466),
}

TOLERANCE = 0.1  # absolute, or 1% relative for large z


def check_reproduction():
    """Compare against REFERENCE_Z and report every disagreement."""
    failures = []
    print(f"{'model':10s} {'corpus':18s} {'':>8s} {'this run':>10s} "
          f"{'reference':>10s} {'delta':>8s}")
    for (model_key, corpus_key), reference in REFERENCE_Z.items():
        current = z_scores[model_key, corpus_key]
        for value, expected, name in zip(current, reference, ("z(0.25)", "z(0.50)")):
            delta = value - expected
            tolerance = max(TOLERANCE, 0.01 * abs(expected))
            ok = abs(delta) <= tolerance
            marker_agrees = (value > Z_THRESHOLD) == (expected > Z_THRESHOLD)
            if not (ok and marker_agrees):
                failures.append((model_key, corpus_key, name, value, expected))
            print(
                f"{model_key:10s} {corpus_key:18s} {name:>8s} {value:10.3f} "
                f"{expected:10.3f} {delta:+8.3f} "
                f"{'' if ok else '  OUT OF TOLERANCE'}"
                f"{'' if marker_agrees else '  Z>3 MARKER DIFFERS'}"
            )
    print()
    if failures:
        print(f"{len(failures)} of {2 * len(REFERENCE_Z)} z-scores disagree. "
              f"Investigate before using this run.")
    else:
        print(f"All {2 * len(REFERENCE_Z)} z-scores reproduce within tolerance.")
    return failures


failures = check_reproduction()